# S6E11 Fraud: GLM Boost on Ten Folds

Thirteenth notebook in the series. In the [pooled stack](https://www.kaggle.com/code/sameershaikh27/s6e11-fraud-pooled-stack-of-public-oof-banks) of 89 members from six authors, the single largest coefficient went to an unremarkable model of mine: a plain 4-leaf tree trained on **ten folds** instead of five. The reason is not the trees. With ten folds, every encoding is computed from 90% of the labels instead of 80%, and on columns where the median value occurs in three rows that is a different feature. No public bank contains a ten-fold member, so a ten-fold version of the strongest recipe, the [GLM boost of notebook 11](https://www.kaggle.com/code/sameershaikh27/s6e11-fraud-glm-boost-seed-bag-and-full-refit), is the one member the stack cannot get anywhere else.

Same recipe as notebook 11, three changes: ten folds, two encoding seeds (ten fits each plus a refit), and out-of-fold predictions that still cover every row once, so they pool with the five-fold banks directly.

The three ideas the recipe rests on, each measured by its author:

| idea | who measured it | effect |
|---|---|---|
| **Boost trees from a GLM.** Fit a logistic regression on the encoding logits, then let 4-leaf trees fit only its residual via `init_score`. | megayak; goodpjw2008 Part 6 | single model 0.9504 → **0.9514** |
| **Bag over encoding seeds.** Re-draw the target encoding's inner split, not only the model seed, and average. | areebbb; goodpjw2008 Part 4 | 0.95037 → 0.95077 pooled; +0.0001 to +0.0002 per member |
| **Full-data refit for test.** One more model on all 700k rows, encodings from all rows, 1.25x the folds' mean trees. OOF unchanged. | goodpjw2008 Part 4; areebbb; megayak | +0.0001 to +0.0002 public LB every time |

My [seed study](https://www.kaggle.com/code/sameershaikh27/s6e11-fraud-seed-and-fold-averaging-measured) found almost nothing from seeds because it varied only the model seed. The encoding's inner split is where the variance lives.

This notebook implements all three on the series' folds. Everything is my own code; the ideas are credited where used.

## Credits
| Whose work | What I took |
|---|---|
| **megayak**, [Per-Value + Digit TE, Shallow-Tree Stack](https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack) | the GLM-then-trees idea and its picture of the generator: label first, every field written given the label, so the signal is additive plus a few interactions |
| **goodpjw2008**, [Part 6: GLM Boost + Stack](https://www.kaggle.com/code/goodpjw2008/s6e11-part-6-glm-boost-stack-lb-0-95215) | the joint keys, key counts as tree inputs, the seed-bag protocol, the 1.25x refit rule, and the plain-GLM finding of Part 5 |
| **areebbb**, [Pooled Stack + Fusion](https://www.kaggle.com/code/areebbb/s6e11-pooled-stack-fusion-lb-0-95196) | encoding-seed bagging measured fold by fold |

## 1. Setup and keys

In [ ]:
import os, glob, gc, time, warnings
import numpy as np, pandas as pd, lightgbm as lgb
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
warnings.filterwarnings("ignore")
SEED, N_FOLDS, TARGET = 42, 10, "is_fraud"
BAG = [(0, 42), (1, 137)]                     # (encoding seed, model seed)
LR, LEAVES = 0.02, 4
def find(name, hint=""):
    hits = sorted(glob.glob(os.path.join("/kaggle/input", "**", name), recursive=True))
    hits = [h for h in hits if hint in h] or hits; assert hits, name; return hits[0]
tr = pd.read_csv(find("train.csv", "s6e11")); te = pd.read_csv(find("test.csv", "s6e11"))
CATS = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
FLAGS = ["is_foreign_transaction", "is_new_merchant", "used_vpn", "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt"]
NUMS = ["amount_usd", "hours_since_last_txn", "txn_count_last_24h", "distance_from_home_km", "card_age_months", "customer_age",
        "account_balance_usd", "cvv_retry_count", "velocity_score", "time_of_day_hour", "day_of_week", "merchant_risk_score", "prior_disputes"]
FLOATS = ["amount_usd", "hours_since_last_txn", "distance_from_home_km", "account_balance_usd", "velocity_score", "merchant_risk_score"]
FEATS = NUMS + FLAGS + CATS
for d in (tr, te): d[FLAGS] = d[FLAGS].astype("int8")
y = tr[TARGET].astype(int).values; NTR, NTE = len(tr), len(te)
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(NTR), y))
full = pd.concat([tr[FEATS], te[FEATS]], ignore_index=True)

# --- integer key matrix: 24 values + 36 digit keys + 7 joint keys ---
def q10(s): return pd.qcut(s, 10, labels=False, duplicates="drop").astype(int)
keys = {}
for c in FEATS: keys[f"v_{c}"] = pd.factorize(full[c])[0]
for c in FLOATS:
    v = full[c].values; i = np.floor(v).astype(np.int64); s = pd.Series(i).astype(str)
    keys[f"cents_{c}"] = (np.round(v*100) % 100).astype(np.int64)
    keys[f"int_{c}"] = i; keys[f"ten_{c}"] = np.floor(v/10).astype(np.int64); keys[f"dec_{c}"] = np.floor(v*10 + 1e-6).astype(np.int64)
    keys[f"l1_{c}"] = (s.str.len()*10 + s.str[0].astype(int)).values; keys[f"l2_{c}"] = (s.str.len()*100 + s.str[:2].astype(int)).values
JOINT = {"j_cat_risk": ("merchant_category", "merchant_risk_score"), "j_foreign_ip": ("is_foreign_transaction", "ip_country_mismatch"),
         "j_cat_amount": ("merchant_category", "q:amount_usd"), "j_txn_velocity": ("txn_count_last_24h", "q:velocity_score"),
         "j_cat_age": ("merchant_category", "customer_age"), "j_hour_dow": ("time_of_day_hour", "day_of_week"), "j_txn_age": ("txn_count_last_24h", "customer_age")}
for name, cols in JOINT.items():
    k = np.zeros(len(full), np.int64)
    for c in cols:
        u = pd.factorize(q10(full[c[2:]]) if c.startswith("q:") else full[c])[0]; k = k*(u.max()+1) + u
    keys[name] = k
KNAMES = list(keys)
KMAT = np.column_stack([pd.factorize(keys[k])[0] for k in KNAMES]).astype(np.int32)    # dense codes per key
NCAT = KMAT.max(axis=0) + 1
KCNT = np.column_stack([np.bincount(KMAT[:, j])[KMAT[:, j]] for j in range(KMAT.shape[1])]).astype(np.float32)  # rows behind each key
CENTS = np.column_stack([keys[f"cents_{c}"] for c in FLOATS]).astype(np.float32)
print(f"train {NTR:,}  test {NTE:,}  keys {len(KNAMES)}  (values {len(FEATS)}, digit {6*len(FLOATS)}, joint {len(JOINT)})")

## 2. Target encoding engine

Nested, as in every notebook of the series: training rows get rates from an inner 5-fold split whose seed is the **encoding seed**; validation and test rows get rates from the whole training fold. `encode(None, ...)` encodes all 700k rows the same way for the full-data refit.

In [ ]:
def te_rate(k, t, ncat, prior, smooth):
    s = np.bincount(k, weights=t, minlength=ncat); n = np.bincount(k, minlength=ncat)
    return (s + prior*smooth) / (n + smooth)

def encode(a, b, smooth=20.0, inner=5, teseed=0):
    """a: training row indices (None = all train rows). b: held-out row indices (None = none).
    Returns encodings for rows a (out of fold), rows b, and test rows."""
    a = np.arange(NTR) if a is None else a
    ya = y[a].astype(np.float64); Ka = KMAT[a]; Kt = KMAT[NTR:]; Kb = KMAT[b] if b is not None else None
    Ea = np.empty((len(a), KMAT.shape[1]), np.float32); Et = np.empty((NTE, KMAT.shape[1]), np.float32)
    Eb = np.empty((len(b), KMAT.shape[1]), np.float32) if b is not None else None
    inner_f = list(StratifiedKFold(inner, shuffle=True, random_state=teseed).split(np.zeros(len(a)), ya))
    for j in range(KMAT.shape[1]):
        for i, o in inner_f:
            Ea[o, j] = te_rate(Ka[i, j], ya[i], NCAT[j], ya[i].mean(), smooth)[Ka[o, j]]
        r = te_rate(Ka[:, j], ya, NCAT[j], ya.mean(), smooth)
        Et[:, j] = r[Kt[:, j]]
        if b is not None: Eb[:, j] = r[Kb[:, j]]
    return Ea, Eb, Et

def logit(p):
    p = np.clip(p, 1e-6, 1-1e-6); return np.log(p/(1-p))
t0 = time.time(); _ = encode(FOLDS[0][0], FOLDS[0][1]); print(f"one outer fold encoded in {time.time()-t0:.0f}s")

## 3. The GLM

A **plain** design, following goodpjw2008's Part 5 finding that splines and one-hot values only hurt once the encodings are in: standardized logits of all 67 encodings, standardized numerics (log for money and hours), the flags, one-hot categoricals. Newton-Cholesky solver, C = 0.3.

In [ ]:
num = full[NUMS].astype(float).copy()
for c in ("amount_usd", "account_balance_usd", "hours_since_last_txn"): num[c] = np.log1p(num[c])
Z = StandardScaler().fit_transform(num).astype(np.float32)
OH = OneHotEncoder(sparse_output=False, dtype=np.float32).fit_transform(full[CATS].astype(str))
BASE = np.hstack([Z, full[FLAGS].values.astype(np.float32), OH])

def glm_logits(a, Ea, b, Eb, Et):
    sc = StandardScaler().fit(logit(Ea))
    Xa = np.hstack([sc.transform(logit(Ea)), BASE[a]])
    m = LogisticRegression(C=0.3, max_iter=300, tol=1e-7, solver="newton-cholesky").fit(Xa, y[a])
    ga = m.decision_function(Xa)
    gt = m.decision_function(np.hstack([sc.transform(logit(Et)), BASE[NTR:]]))
    gb = m.decision_function(np.hstack([sc.transform(logit(Eb)), BASE[b]])) if b is not None else None
    return ga, gb, gt

## 4. Trees that start from the GLM

LightGBM with 4 leaves, learning rate 0.02, `init_score` = the GLM's logit, so every tree fits the residual. Tree inputs: the raw columns, the cents, the encodings, and **how many rows stand behind each key** (a rare value's encoding deserves less trust, and the tree can learn that). Predictions are `raw_score + GLM logit`.

In [ ]:
RAW = full[FEATS].copy()
for c in CATS: RAW[c] = pd.Categorical(RAW[c], categories=sorted(RAW[c].unique()))
def tree_frame(idx, E):
    D = RAW.iloc[idx].reset_index(drop=True).copy()
    for j, c in enumerate(FLOATS): D[f"cents_{c}"] = CENTS[idx, j]
    for j, k in enumerate(KNAMES): D[f"te_{k}"] = E[:, j]; D[f"cnt_{k}"] = KCNT[idx, j]
    return D
PARAMS = dict(objective="binary", metric="auc", learning_rate=LR, num_leaves=LEAVES, min_child_samples=100, feature_fraction=0.5,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, max_bin=255, verbosity=-1, num_threads=4)

def boost_fold(a, b, teseed, mseed):
    Ea, Eb, Et = encode(a, b, teseed=teseed)
    ga, gb, gt = glm_logits(a, Ea, b, Eb, Et)
    Xa, Xb, Xt = tree_frame(a, Ea), tree_frame(b, Eb), tree_frame(np.arange(NTR, NTR+NTE), Et)
    dtr = lgb.Dataset(Xa, y[a], init_score=ga); dva = lgb.Dataset(Xb, y[b], init_score=gb, reference=dtr)
    m = lgb.train({**PARAMS, "seed": mseed}, dtr, num_boost_round=20000, valid_sets=[dva], callbacks=[lgb.early_stopping(300, verbose=False)])
    pb = m.predict(Xb, raw_score=True, num_iteration=m.best_iteration) + gb
    pt = m.predict(Xt, raw_score=True, num_iteration=m.best_iteration) + gt
    return pb, pt, gb, m.best_iteration

def boost_refit(n_rounds, teseed, mseed):
    Ea, _, Et = encode(None, None, teseed=teseed)
    a = np.arange(NTR); ga, _, gt = glm_logits(a, Ea, None, None, Et)
    Xa, Xt = tree_frame(a, Ea), tree_frame(np.arange(NTR, NTR+NTE), Et)
    m = lgb.train({**PARAMS, "seed": mseed}, lgb.Dataset(Xa, y, init_score=ga), num_boost_round=n_rounds)
    return m.predict(Xt, raw_score=True) + gt

## 5. Bag: two encoding seeds, ten folds each, then a refit of each

For each seed: ten fold models give the OOF prediction and a fold-average test prediction; one more model on all rows, with 1.11x the mean best iteration (11% more rows than a 90% fold), gives the refit test prediction. The GLM's own OOF is kept too, so the gain from boosting is visible per fold.

In [ ]:
OOF, GLM_OOF, FOLDAVG, REFIT, LOG = {}, {}, {}, {}, []
t0 = time.time()
for teseed, mseed in BAG:
    name = f"gb10_s{teseed}"; OOF[name] = np.zeros(NTR); GLM_OOF[name] = np.zeros(NTR); FOLDAVG[name] = np.zeros(NTE); its = []
    for f, (a, b) in enumerate(FOLDS):
        pb, pt, gb, it = boost_fold(a, b, teseed, mseed)
        OOF[name][b] = pb; GLM_OOF[name][b] = gb; FOLDAVG[name] += pt/N_FOLDS; its.append(it)
        LOG.append((name, f, roc_auc_score(y[b], gb), roc_auc_score(y[b], pb), it))
        print(f"{name} fold {f}: GLM {LOG[-1][2]:.5f} -> boosted {LOG[-1][3]:.5f}  trees {it}  ({time.time()-t0:.0f}s)")
        gc.collect()
    n = int(round(np.mean(its)*1.11)); REFIT[name] = boost_refit(n, teseed, mseed)
    print(f"==> {name}: GLM OOF {roc_auc_score(y, GLM_OOF[name]):.5f} | boosted OOF {roc_auc_score(y, OOF[name]):.5f} | refit with {n} trees  ({time.time()-t0:.0f}s)\n")
    np.save(f"{name}_oof.npy", OOF[name].astype(np.float32)); np.save(f"{name}_test.npy", FOLDAVG[name].astype(np.float32)); np.save(f"{name}_full.npy", REFIT[name].astype(np.float32))

## 6. The accounting

In [ ]:
names = list(OOF)
bag_oof = np.mean([rankdata(OOF[n])/NTR for n in names], axis=0)
bag_full = np.mean([rankdata(REFIT[n])/NTE for n in names], axis=0)
bag_foldavg = np.mean([rankdata(FOLDAVG[n])/NTE for n in names], axis=0)
res = pd.DataFrame({"GLM OOF": {n: roc_auc_score(y, GLM_OOF[n]) for n in names}, "boosted OOF": {n: roc_auc_score(y, OOF[n]) for n in names}})
res["gain x1e-4"] = (res["boosted OOF"] - res["GLM OOF"])*1e4
print(res.round(5).to_string())
print(f"\nseed mean {res['boosted OOF'].mean():.5f}   seed spread {res['boosted OOF'].max()-res['boosted OOF'].min():.5f}")
print(f"{len(names)}-seed rank bag OOF AUC: {roc_auc_score(y, bag_oof):.5f}")
print(f"reference: notebook 11 five-fold GLM boost: seeds 0.95121-0.95133, 3-seed bag 0.95136 | notebook 8 ten-fold plain tree 0.95064")
np.save("gb10_bag_oof.npy", bag_oof.astype(np.float32)); np.save("gb10_bag_full.npy", bag_full.astype(np.float32)); np.save("gb10_bag_test.npy", bag_foldavg.astype(np.float32))
pd.DataFrame(LOG, columns=["seed", "fold", "glm_auc", "boost_auc", "trees"]).to_csv("fold_log.csv", index=False)

## 7. Submissions

Two files: the bag's **refit** test prediction (the recommended one) and its **fold-average** test prediction, so the refit effect is measured once more on the board.

In [ ]:
pd.DataFrame({"id": te["id"], TARGET: bag_full}).to_csv("submission.csv", index=False)
pd.DataFrame({"id": te["id"], TARGET: bag_foldavg}).to_csv("submission_foldavg.csv", index=False)
pd.DataFrame({"id": tr["id"], TARGET: y, **{n: OOF[n] for n in names}, "bag": bag_oof}).to_csv("oof.csv", index=False)
pd.DataFrame({"id": te["id"], **{f"{n}_full": REFIT[n] for n in names}, "bag_full": bag_full, "bag_foldavg": bag_foldavg}).to_csv("test_preds.csv", index=False)
print("written: submission.csv (refit), submission_foldavg.csv, oof.csv, test_preds.csv, per-seed npy")

## 8. Results log

| notebook | model | OOF AUC | public LB |
|---|---|---|---|
| [notebook 3](https://www.kaggle.com/code/sameershaikh27/s6e11-fraud-digit-key-te-4-leaf-trees-blend) | best single tree, digit-key TE | 0.95034 | 0.95089 |
| [notebook 10](https://www.kaggle.com/code/sameershaikh27/s6e11-fraud-the-playbook-10-notebooks-in-1-page) | blend of nine members | 0.95087 | 0.95123 |
| this, fold average | GLM boost, ten folds, 2 seeds | 0.95159 | not yet submitted (daily limit) |
| this, refit | same, test from full-data refits | 0.95159 | **0.95118** |

Per seed: GLM 0.95082 to 0.95085, boosted 0.95150 and 0.95160, two-seed rank bag **0.95159**, the strongest single member in the pooled bank (goodpjw2008's five-seed five-fold bag: 0.95151). Ten folds added 0.0002 over notebook 11's five. On the public board the refit scored 0.95118, 0.0004 below its own OOF, the fourth GLM-centred single model in this competition to land below expectation on the 20% public sample. Its job is inside the pooled stack of [notebook 12](https://www.kaggle.com/code/sameershaikh27/s6e11-fraud-pooled-stack-of-public-oof-banks).

## 9. What ten folds change

- **More labels per encoding, not more trees.** The gain over notebook 11 comes from encodings fitted on 90% of the rows. The tree count barely moves.
- **A ten-fold OOF pools with five-fold OOFs.** Every row is still predicted exactly once by a model that never saw its label, so the stacker treats it like any other member.
- **It is the member nobody else has.** In a stack of 89 public members the marginal value of one more five-fold tree is near zero; a different fold geometry is a different view.
- Predictions are saved per seed and as a bag, in the same file layout as every public bank.